In [ ]:
# ============================================
# PNEUMOSCAN — Stage-1 (Normal vs Pneumonia) Evaluator with UI Button
# Uses your saved model and your 200-image test split
# ============================================

# ---- Imports ----
import os, numpy as np, matplotlib.pyplot as plt
from itertools import product
from IPython.display import display, clear_output
import ipywidgets as widgets

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import load_model

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_fscore_support
)

# ---- 1) Mount Drive ----
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print("Drive mount note:", e)

# ---- 2) Fixed Paths ----
MODEL_PATH = "/content/drive/MyDrive/pneumonia_detection_system/model/efficientnetb0_stage1_norm_vs_pneu_fixed.keras"
TEST_DIR   = "/content/drive/MyDrive/stage1_dataset_splitted/test"   # contains Normal (100) & Pneumonia (100)

IMG_SIZE   = (224, 224)
BATCH_SIZE = 16
CLASSES    = ['Normal', 'Pneumonia']  # class order consistent with your training script

# ---- UI Elements ----
hdr = widgets.HTML("<h3> Pneumonia_Detection_System — Stage-1 Model Evaluator (Normal vs Pneumonia)</h3>")
status_box = widgets.HTML("<b>Status:</b> Initializing...")
evaluate_btn = widgets.Button(description="Evaluate", button_style='primary', tooltip="Run evaluation on the 200-image test set")
evaluate_btn.layout.width = "200px"
box = widgets.VBox([hdr, status_box, evaluate_btn])
display(box)

# Globals to hold objects across button click
_model = None
_test_gen = None
_class_indices = None

# ---- 3) Load Model ----
if not os.path.exists(MODEL_PATH):
    status_box.value = f"<b>Status:</b>  Model not found at: <code>{MODEL_PATH}</code>"
    raise SystemExit

status_box.value = "<b>Status:</b> Loading model…"
_model = load_model(MODEL_PATH)
status_box.value = "<b>Status:</b>  Model loaded."

# ---- 4) Load Test Dataset (200 images) ----
if not os.path.isdir(TEST_DIR):
    status_box.value = f"<b>Status:</b>  Test folder not found at: <code>{TEST_DIR}</code>"
    raise SystemExit

status_box.value = "<b>Status:</b> Loading 200-image test set…"

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
_test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    classes=CLASSES
)

_class_indices = _test_gen.class_indices  # {'Normal':0,'Pneumonia':1}
status_box.value = (f"<b>Status:</b>  Test set ready "
                    f"(Normal={(_test_gen.classes==_class_indices['Normal']).sum()}, "
                    f"Pneumonia={(_test_gen.classes==_class_indices['Pneumonia']).sum()}). "
                    f"<br> Click <b>Evaluate</b> to compute metrics.")

# ---- 5) Helper: Pretty Confusion Matrix Plot ----
def plot_confusion_matrix(cm, class_names):
    fig = plt.figure(figsize=(5, 4), dpi=120)
    plt.imshow(cm, interpolation='nearest')
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    thresh = cm.max() / 2.0 if cm.size else 0.5
    for i, j in product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    return fig

# ---- 6) Evaluation on Button Click ----
out_area = widgets.Output()
display(out_area)

def on_evaluate_clicked(b):
    with out_area:
        clear_output(wait=True)
        if _model is None or _test_gen is None:
            print("Model/Test generator not initialized.")
            return

        print(" Running evaluation …")

        # a) Evaluate() for Loss/Accuracy
        loss, acc = _model.evaluate(_test_gen, verbose=0)
        print(f"\n=== Overall Metrics ===")
        print(f"Test Loss     : {loss:.5f}")
        print(f"Test Accuracy : {acc*100:.2f}%")

        # b) Predictions & Confusion Matrix
        y_true = _test_gen.classes
        y_prob = _model.predict(_test_gen, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)

        # Ensure label order [Normal, Pneumonia]
        normal_idx     = _class_indices['Normal']
        pneumonia_idx  = _class_indices['Pneumonia']
        cm = confusion_matrix(y_true, y_pred, labels=[normal_idx, pneumonia_idx])

        # cm layout (with labels=[Normal, Pneumonia]):
        # [[TN, FP],
        #  [FN, TP]]
        TN, FP, FN, TP = cm.ravel()

        # c) Derived Metrics
        eps = 1e-12

        # Metrics when Pneumonia is Positive Class
        sensitivity_pneumonia = TP / (TP + FN + eps)  # Recall of Pneumonia
        specificity_pneumonia = TN / (TN + FP + eps)  # Specificity wrt Normal
        precision_pneumonia = TP / (TP + FP + eps)
        recall_pneumonia = sensitivity_pneumonia
        f1_pneumonia = 2 * precision_pneumonia * recall_pneumonia / (precision_pneumonia + recall_pneumonia + eps)

        # Metrics when Normal is Positive Class (NEW)
        sensitivity_normal = TN / (TN + FP + eps)     # Recall of Normal
        specificity_normal = TP / (TP + FN + eps)     # Specificity wrt Pneumonia
        precision_normal = TN / (TN + FN + eps)
        recall_normal = sensitivity_normal
        f1_normal = 2 * precision_normal * recall_normal / (precision_normal + recall_normal + eps)

        # Macro/Weighted P/R/F1 for completeness
        prfs = precision_recall_fscore_support(y_true, y_pred, labels=[normal_idx, pneumonia_idx], zero_division=0)
        precision_macro = prfs[0].mean()
        recall_macro    = prfs[1].mean()
        f1_macro        = prfs[2].mean()

        # d) AUC-ROC for BOTH classes (IMPROVED)
        # AUC for Pneumonia (as positive class)
        y_true_pneumonia = (y_true == pneumonia_idx).astype(int)
        pneumonia_scores = y_prob[:, pneumonia_idx]
        try:
            auc_pneumonia = roc_auc_score(y_true_pneumonia, pneumonia_scores)
        except ValueError:
            auc_pneumonia = float('nan')

        # AUC for Normal (as positive class) - NEW
        y_true_normal = (y_true == normal_idx).astype(int)
        normal_scores = y_prob[:, normal_idx]
        try:
            auc_normal = roc_auc_score(y_true_normal, normal_scores)
        except ValueError:
            auc_normal = float('nan')

        # Average AUC (Macro-average) - NEW
        if not np.isnan(auc_pneumonia) and not np.isnan(auc_normal):
            auc_macro = (auc_pneumonia + auc_normal) / 2
        else:
            auc_macro = float('nan')

        # e) Classification Report
        print("\n=== Classification Report (per class) ===")
        print(classification_report(
            y_true, y_pred,
            target_names=['Normal', 'Pneumonia'],
            labels=[normal_idx, pneumonia_idx],
            digits=4
        ))

        # f) Confusion Matrix (counts + TP/TN/FP/FN)
        print("\n=== Confusion Matrix (counts) ===")
        print(cm)
        print(f"\nTP: {TP} ")
        print(f"TN: {TN}  ")
        print(f"FP: {FP} ")
        print(f"FN: {FN}  ")

        # g) Key Metrics Summary (BOTH classes) - IMPROVED
        print("\n" + "="*50)
        print("KEY METRICS SUMMARY")
        print("="*50)

        print("\n--- For PNEUMONIA (as Positive Class) ---")
        print(f"Precision          : {precision_pneumonia:.4f}")
        print(f"Recall/Sensitivity : {sensitivity_pneumonia:.4f}")
        print(f"F1-Score          : {f1_pneumonia:.4f}")
        print(f"AUC-ROC           : {auc_pneumonia:.4f}")

        print("\n--- For NORMAL (as Positive Class) ---")
        print(f"Precision          : {precision_normal:.4f}")
        print(f"Recall/Sensitivity : {sensitivity_normal:.4f}")
        print(f"F1-Score          : {f1_normal:.4f}")
        print(f"AUC-ROC           : {auc_normal:.4f}")

        print("\n--- General Metrics ---")
        print(f"Specificity (for Pneumonia as positive): {specificity_pneumonia:.4f}")
        print(f"Specificity (for Normal as positive)  : {specificity_normal:.4f}")
        print(f"Macro-average AUC                     : {auc_macro:.4f}")

        print("\n=== Macro Averages (Normal & Pneumonia) ===")
        print(f"Precision (macro): {precision_macro:.4f}")
        print(f"Recall (macro)   : {recall_macro:.4f}")
        print(f"F1 (macro)       : {f1_macro:.4f}")

        # h) Visuals — Confusion Matrix Heatmap
        fig_cm = plot_confusion_matrix(cm, ['Normal', 'Pneumonia'])
        plt.show(fig_cm)

        # i) ROC Curves for BOTH classes (IMPROVED)
        print("\n" + "="*50)
        print("ROC CURVES (One-vs-Rest)")
        print("="*50)

        # Create subplot for both ROC curves
        fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=120)

        # ROC Curve for Normal
        y_true_normal = (y_true == normal_idx).astype(int)
        normal_scores = y_prob[:, normal_idx]
        try:
            fpr_norm, tpr_norm, _ = roc_curve(y_true_normal, normal_scores)
            auc_norm = auc_normal
            axes[0].plot(fpr_norm, tpr_norm, label=f'ROC curve (AUC = {auc_norm:.4f})',
                        linewidth=2, color='green')
            axes[0].plot([0, 1], [0, 1], linestyle='--', linewidth=1, color='gray')
            axes[0].set_xlim([0.0, 1.0])
            axes[0].set_ylim([0.0, 1.05])
            axes[0].set_xlabel('False Positive Rate')
            axes[0].set_ylabel('True Positive Rate (Sensitivity)')
            axes[0].set_title('ROC Curve — Normal\n(As Positive Class)')
            axes[0].legend(loc="lower right")
            axes[0].grid(True, alpha=0.3)
        except Exception as e:
            axes[0].text(0.5, 0.5, f"ROC unavailable\n{e}",
                        ha='center', va='center', transform=axes[0].transAxes)
            axes[0].set_title('ROC Curve — Normal')

        # ROC Curve for Pneumonia
        y_true_pneumonia = (y_true == pneumonia_idx).astype(int)
        pneumonia_scores = y_prob[:, pneumonia_idx]
        try:
            fpr_pneu, tpr_pneu, _ = roc_curve(y_true_pneumonia, pneumonia_scores)
            auc_pneu = auc_pneumonia
            axes[1].plot(fpr_pneu, tpr_pneu, label=f'ROC curve (AUC = {auc_pneu:.4f})',
                        linewidth=2, color='orange')
            axes[1].plot([0, 1], [0, 1], linestyle='--', linewidth=1, color='gray')
            axes[1].set_xlim([0.0, 1.0])
            axes[1].set_ylim([0.0, 1.05])
            axes[1].set_xlabel('False Positive Rate')
            axes[1].set_ylabel('True Positive Rate (Sensitivity)')
            axes[1].set_title('ROC Curve — Pneumonia\n(As Positive Class)')
            axes[1].legend(loc="lower right")
            axes[1].grid(True, alpha=0.3)
        except Exception as e:
            axes[1].text(0.5, 0.5, f"ROC unavailable\n{e}",
                        ha='center', va='center', transform=axes[1].transAxes)
            axes[1].set_title('ROC Curve — Pneumonia')

        plt.tight_layout()
        plt.show()

        # Additional single ROC curve for comparison
        print("\n--- Combined ROC Analysis ---")
        fig_single = plt.figure(figsize=(5, 4), dpi=120)
        try:
            # Plot both curves on same graph for comparison
            fpr_norm, tpr_norm, _ = roc_curve(y_true_normal, normal_scores)
            fpr_pneu, tpr_pneu, _ = roc_curve(y_true_pneumonia, pneumonia_scores)

            plt.plot(fpr_norm, tpr_norm, label=f'Normal (AUC = {auc_normal:.4f})',
                    linewidth=2, color='green')
            plt.plot(fpr_pneu, tpr_pneu, label=f'Pneumonia (AUC = {auc_pneumonia:.4f})',
                    linewidth=2, color='orange')
            plt.plot([0, 1], [0, 1], linestyle='--', linewidth=1, color='gray')
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate (Sensitivity)')
            plt.title('ROC Curves Comparison\n(Normal vs Pneumonia)')
            plt.legend(loc="lower right")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Combined ROC curve skipped: {e}")

        # j) Dataset Statistics and Final Summary
        print("\n" + "="*50)
        print("EVALUATION COMPLETE")
        print("="*50)
        print(f"\nTotal Samples Evaluated: {len(y_true)}")
        print(f"  • Normal: {sum(y_true == normal_idx)} images")
        print(f"  • Pneumonia: {sum(y_true == pneumonia_idx)} images")

        # Final accuracy check
        correct_predictions = (y_true == y_pred).sum()
        calculated_accuracy = correct_predictions / len(y_true)
        print(f"\nManual Accuracy Check: {correct_predictions}/{len(y_true)} = {calculated_accuracy*100:.2f}%")
        print(f"Model.evaluate() Accuracy: {acc*100:.2f}%")

        # Performance interpretation
        print("\n=== Performance Interpretation ===")
        if auc_pneumonia > 0.9:
            print(" Excellent Pneumonia detection capability (AUC > 0.9)")
        elif auc_pneumonia > 0.8:
            print(" Good Pneumonia detection capability (AUC > 0.8)")
        elif auc_pneumonia > 0.7:
            print(" Fair Pneumonia detection capability (AUC > 0.7)")
        else:
            print(" Pneumonia detection needs improvement")

        if acc > 0.9:
            print(" Excellent overall accuracy (> 90%)")
        elif acc > 0.8:
            print(" Good overall accuracy (> 80%)")
        else:
            print(" Overall accuracy needs improvement")

# Only enable button after dataset load
def enable_button_after_load():
    evaluate_btn.disabled = False
    evaluate_btn.button_style = 'success'

evaluate_btn.on_click(on_evaluate_clicked)
enable_button_after_load()

Mounted at /content/drive


Found 200 images belonging to 2 classes.


Output()